# Task 2.1 — Dataset Setup

## Contribution reproduced
We reproduce **shapelet discovery** (Algorithm 1 and Section 2) and the **Fast Shapelet Discovery** (Algorithms 4–6). Evaluation metric: **accuracy** and **information gain**.

---

## Dataset: Synthetic Time Series (Normal vs Anomalous Waves)

We generate a **synthetic time-series dataset** for a **binary classification** problem: **normal waves** (class 0) vs **anomalous waves** (class 1). Each sample is a 1D time series of consecutive time points. We ensure **at least 100 samples** (we use 120 total: 60 per class).

**Why it is a good testbed for Time Series Shapelets:** The Logical-Shapelets method (Mueen et al., KDD 2011) works on a matrix of (samples × time steps) and class labels. It looks for discriminative **subsequences (shapelets)** whose presence or absence (via distance to the shapelet) helps separate the classes. Our synthetic data is designed so that one class has a recurring “normal” wave pattern and the other has an “anomalous” pattern (e.g. different frequency or a spike). This gives the algorithm a chance to find a shapelet that corresponds to the normal or anomalous motif.

**Limitations compared to real-world data:** The paper uses real time series (e.g. UCR: accelerometer, ECG) with natural temporal structure and often clear motifs. Our synthetic data is controlled and may not have the same noise, non-stationarity, or multiple overlapping motifs as real sensors. We do not expect to match the paper’s reported accuracies on UCR; the goal is to verify that the implementation is correct and produces a shapelet and threshold that maximize information gain.

## Generate synthetic dataset (normal vs anomalous waves)

In [4]:
# Reproducibility: fix seed (see Reproducibility Checklist in task 2 3)
RANDOM_STATE = 42
import numpy as np

# We want at least 100 samples; we use 120 (60 per class)
n_samples_per_class = 60
n_timesteps = 50
np.random.seed(RANDOM_STATE)

# Time axis (consecutive time points)
t = np.linspace(0, 4 * np.pi, n_timesteps)

# Class 0: "normal" waves — sine with small noise
normal_list = []
for i in range(n_samples_per_class):
    noise = np.random.randn(n_timesteps) * 0.2
    wave = np.sin(t) + noise
    normal_list.append(wave)

# Class 1: "anomalous" waves — different pattern (e.g. higher frequency + spike)
anomalous_list = []
for i in range(n_samples_per_class):
    noise = np.random.randn(n_timesteps) * 0.2
    # Different frequency and a small spike in the middle
    wave = np.sin(2 * t) + 0.5 * np.sin(4 * t) + noise
    spike_pos = n_timesteps // 2
    wave[spike_pos] += 1.5
    anomalous_list.append(wave)

# Stack into one matrix: rows = samples, columns = time points
X_normal = np.array(normal_list)
X_anomalous = np.array(anomalous_list)
X = np.vstack([X_normal, X_anomalous])
y = np.array([0] * n_samples_per_class + [1] * n_samples_per_class)

print('Dataset shape:', X.shape)
print('Class counts:', np.bincount(y))
print('Number of samples (must be >= 100):', len(y))

Dataset shape: (120, 50)
Class counts: [60 60]
Number of samples (must be >= 100): 120


The dataset has **consecutive time points** (no gaps). Class 0 is a simple sine wave with noise; class 1 has a different shape (higher-frequency components and a spike). This gives the shapelet algorithm a discriminative subsequence to find (e.g. the spike or the smoother sine segment).

## Train/test split and save to partB/data/

In [5]:
from sklearn.model_selection import train_test_split
import os

# 80/20 train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Train class counts:', np.bincount(y_train), 'Test class counts:', np.bincount(y_test))

# Save to partB/data/ so other notebooks can load it
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)
np.savez(
    os.path.join(data_dir, 'toy_ts.npz'),
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    X=X, y=y
)
print('Saved dataset to', data_dir, '/ toy_ts.npz')

Train shape: (96, 50) Test shape: (24, 50)
Train class counts: [48 48] Test class counts: [12 12]
Saved dataset to data / toy_ts.npz


We do not apply global scaling here; the shapelet code **z-normalizes each subsequence** when computing distances (paper Section 2, Eq. 2).

## Load script: how to load the saved dataset

In [6]:
# Brief script to load the dataset from partB/data/
# Run this from the partB/ directory.

def load_toy_dataset(data_path='data/toy_ts.npz'):
    """Load the synthetic time series dataset. Returns X_train, X_test, y_train, y_test."""
    data = np.load(data_path)
    return (
        data['X_train'], data['X_test'],
        data['y_train'], data['y_test']
    )

# Example: load and check
X_train_loaded, X_test_loaded, y_train_loaded, y_test_loaded = load_toy_dataset()
print('Loaded train:', X_train_loaded.shape, 'test:', X_test_loaded.shape)

Loaded train: (96, 50) test: (24, 50)
